# Superfermion Benchmark Suite — Source of Truth

Every public benchmark claim, regenerated live in one run. No stored or
historical results are read — every number below is produced by this
notebook, on this machine, in this session.

**Methodology:** identical circuits on both frameworks, 15 trials per
configuration, medians with IQRs, per-size warmups, GC disabled during
timing, correctness gates (statevector fidelity vs Qiskit, TVD vs exact
distributions, cross-checked gradients vs PennyLane and finite differences).

**Competitors:** Qiskit 2.x / Aer, PennyLane Lightning (adjoint).

Timing sections produce `benchmark_data.json`; the transpilation section
re-runs `run_benchpress.py` and refreshes `benchpress_post_gaps.json`.


In [ ]:
import gc
import json
import platform
import subprocess
import time
from pathlib import Path
from statistics import median

import numpy as np

import superfermion as sf
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import qiskit, qiskit_aer

# ── Platform metadata (recorded with every run) ──────────
cpu_model = 'unknown'
n_cores = platform.machine()
try:
    for line in Path('/proc/cpuinfo').read_text().splitlines():
        if line.startswith('model name') and cpu_model == 'unknown':
            cpu_model = line.split(':', 1)[1].strip()
        if line.startswith('cpu cores'):
            n_cores = line.split(':', 1)[1].strip()
except Exception:
    pass

import multiprocessing
PLATFORM = {
    'run_date': time.strftime('%Y-%m-%d %H:%M %Z'),
    'cpu_model': cpu_model,
    'physical_cores': n_cores,
    'logical_cores': multiprocessing.cpu_count(),
    'machine': platform.machine(),
    'os': f'{platform.system()} {platform.release()}',
    'python': platform.python_version(),
    'superfermion': sf.__version__,
    'qiskit': qiskit.__version__,
    'qiskit_aer': qiskit_aer.__version__,
}

import pennylane as qml
from importlib.metadata import version as _pkg_version
PLATFORM['pennylane'] = qml.__version__
PLATFORM['pennylane_lightning'] = _pkg_version('pennylane-lightning')

for k, v in PLATFORM.items():
    print(f'{k:>22}: {v}')

In [ ]:
N_TRIALS = 15

def timed(fn):
    gc.disable()
    t0 = time.perf_counter()
    result = fn()
    elapsed = (time.perf_counter() - t0) * 1000
    gc.enable()
    return result, elapsed

def benchmark(fn, n_trials=N_TRIALS):
    times = []
    for _ in range(n_trials):
        _, t = timed(fn)
        times.append(round(t, 2))
    med = median(times)
    s = sorted(times)
    q1 = s[len(s)//4]
    q3 = s[3*len(s)//4]
    return med, times, q1, q3

def fidelity(sv1, sv2):
    return float(abs(np.vdot(sv1, sv2)) ** 2)

def fmt(sf_ms, other_ms):
    r = other_ms / sf_ms if sf_ms > 0 else float('inf')
    return f'SF {r:.1f}x faster' if r > 1 else f'Other {1/r:.1f}x faster'

print(f'Helpers defined. {N_TRIALS} trials per configuration.')

In [ ]:
def build_hardware_efficient(n, depth=3, seed=42):
    rng = np.random.RandomState(seed)
    angles = rng.uniform(0, 2*np.pi, size=(depth, n, 2))
    sc = sf.Circuit(n)
    qc = QuantumCircuit(n)
    for d in range(depth):
        for q in range(n):
            sc.ry(angles[d, q, 0], q); qc.ry(angles[d, q, 0], q)
            sc.rz(angles[d, q, 1], q); qc.rz(angles[d, q, 1], q)
        for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
        for q in range(1, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    return sc, qc

def build_ghz(n):
    sc = sf.Circuit(n); qc = QuantumCircuit(n)
    sc.h(0); qc.h(0)
    for i in range(1, n): sc.cx(0, i); qc.cx(0, i)
    return sc, qc

def build_clifford_only(n):
    sc = sf.Circuit(n); qc = QuantumCircuit(n)
    for q in range(n):         sc.h(q);  qc.h(q)
    for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    for q in range(n):         sc.s(q);  qc.s(q)
    for q in range(1, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    for q in range(n):         sc.h(q);  qc.h(q)
    for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    return sc, qc

def build_tfim_hamiltonian(n):
    from superfermion.observables.core import Hamiltonian
    from superfermion.chemistry.hamiltonians import PauliString
    terms = []
    for i in range(n - 1):
        ps = ['I'] * n
        ps[i] = 'Z'; ps[i+1] = 'Z'
        terms.append(PauliString(''.join(ps), -1.0))
    for i in range(n):
        ps = ['I'] * n
        ps[i] = 'X'
        terms.append(PauliString(''.join(ps), -0.5))
    return Hamiltonian(terms)

print('Circuit builders defined.')

## 1. Statevector vs Qiskit Aer

Hardware-efficient ansatz (depth 3), exact simulation, n = 10–22.
**Correctness gate:** statevector fidelity vs Aer > 0.9999 at every size.

In [ ]:
sv_qubit_counts = [10, 12, 14, 16, 18, 20, 22]
sv_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print('STATEVECTOR BENCHMARK (Hardware-Efficient Ansatz, depth=3)')
print('=' * 75)

n_max = sv_qubit_counts[-1]
print(f'Warming up at n={n_max}...')
_ = sf.run(build_hardware_efficient(n_max)[0], device='cpu', shots=0)
qc_w = build_hardware_efficient(n_max)[1]; qc_w.save_statevector()
_ = qk_sim.run(qc_w).result()
print('Done.\n')

for n in sv_qubit_counts:
    sc_c, qc_c = build_hardware_efficient(n)
    sf_sv = np.array(sf.run(sc_c, device='cpu', shots=0).statevector)
    qc_c.save_statevector()
    qk_sv = np.array(qk_sim.run(qc_c).result().get_statevector(), dtype=np.complex128)
    fid = fidelity(sf_sv, qk_sv)
    assert fid > 0.9999, f'Fidelity fail n={n}: {fid}'

    _ = sf.run(build_hardware_efficient(n)[0], device='cpu', shots=0)
    qc_wu = build_hardware_efficient(n)[1]; qc_wu.save_statevector()
    _ = qk_sim.run(qc_wu).result()

    def run_sf(n=n):
        sc, _ = build_hardware_efficient(n)
        return sf.run(sc, device='cpu', shots=0)
    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf)

    def run_qk(n=n):
        _, qc = build_hardware_efficient(n)
        qc.save_statevector()
        return qk_sim.run(qc).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk)

    sv_results[n] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'fidelity': fid,
    }
    print(f'n={n:2d} | SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}-{qk_q3:.1f}] | '
          f'fid={fid:.6f} | {fmt(sf_med, qk_med)}')

## 2. Stabilizer vs Qiskit Aer

Clifford circuit, 10k shots, n = 10–500 (word-packed tableau).
**Correctness gate:** TVD of SF samples vs the *exact* statevector distribution,
threshold = 4x the expected sampling TVD.

In [ ]:
stab_qubit_counts = [10, 20, 50, 100, 200, 500]
stab_results = {}
qk_stab = AerSimulator(method='stabilizer')

print('=' * 75)
print('STABILIZER BENCHMARK (Clifford circuit, shots=10000)')
print('=' * 75)

sc_w, qc_w = build_clifford_only(stab_qubit_counts[-1])
print(f'Warming up at n={stab_qubit_counts[-1]}...')
_ = sf.run(sc_w, device='cpu', method='stabilizer', shots=1000)
qc_w_m = qc_w.copy(); qc_w_m.measure_all()
_ = qk_stab.run(qc_w_m, shots=1000).result()
print('Done.\n')

for n in stab_qubit_counts:
    corr = ''
    if n <= 16:
        exact_sv = np.array(sf.run(build_clifford_only(n)[0], device='cpu', shots=0).statevector)
        exact_probs = {format(i, f'0{n}b'): float(p) for i, p in enumerate(np.abs(exact_sv)**2) if p > 1e-15}
        n_check = 200000
        sf_c = sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=n_check).counts
        all_keys = set(sf_c.keys()) | set(exact_probs.keys())
        dist = 0.5 * sum(abs(sf_c.get(k,0)/n_check - exact_probs.get(k,0)) for k in all_keys)
        k = len(exact_probs)
        expected = np.sqrt(k / (2 * np.pi * n_check))
        threshold = expected * 4
        corr = f'TVD={dist:.4f} (expect~{expected:.4f}, thr={threshold:.4f})'
        assert dist < threshold, f'Stabilizer mismatch n={n}: {corr}'

    _ = sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=1000)
    qc_wu = build_clifford_only(n)[1]; qc_wu.measure_all()
    _ = qk_stab.run(qc_wu, shots=1000).result()

    def run_sf_s(n=n):
        return sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=10000)
    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf_s)

    def run_qk_s(n=n):
        _, qc = build_clifford_only(n)
        qc.measure_all()
        return qk_stab.run(qc, shots=10000).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk_s)

    stab_results[n] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'correctness': corr,
    }
    print(f'n={n:3d} | SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}-{qk_q3:.1f}] | {fmt(sf_med, qk_med)} | {corr}')

## 3. MPS vs Qiskit Aer (GHZ)

GHZ circuits (low entanglement, bond dimension 2), 10k shots, SF bond_dim=64.
**Correctness gate:** GHZ distribution ~50/50 on |0...0> and |1...1>.

**Caveat:** GHZ is the trivially-best case for MPS — it demonstrates
framework overhead, not tensor-network scaling on entangled workloads.

In [ ]:
mps_qubit_counts = [10, 20, 30, 50, 80, 100]
mps_results = {}
qk_mps = AerSimulator(method='matrix_product_state')

print('=' * 75)
print('MPS BENCHMARK (GHZ circuit, shots=10000, bond_dim=64)')
print('=' * 75)

sc_w, qc_w = build_ghz(mps_qubit_counts[-1])
print(f'Warming up at n={mps_qubit_counts[-1]}...')
_ = sf.run(sc_w, device='cpu', method='mps', shots=1000, bond_dim=64)
qc_w_m = qc_w.copy(); qc_w_m.measure_all()
_ = qk_mps.run(qc_w_m, shots=1000).result()
print('Done.\n')

for n in mps_qubit_counts:
    corr = ''
    if n <= 50:
        sc_c, _ = build_ghz(n)
        n_check = 100000
        sf_c = sf.run(sc_c, device='cpu', method='mps', shots=n_check, bond_dim=64).counts
        z = '0' * n; o = '1' * n
        total = sum(sf_c.values())
        fz = sf_c.get(z, 0) / total if total > 0 else 0
        fo = sf_c.get(o, 0) / total if total > 0 else 0
        err = abs(fz - 0.5) + abs(fo - 0.5)
        corr = f'GHZ err={err:.4f} (|0>={fz:.3f}, |1>={fo:.3f})'
        assert err < 0.03, f'GHZ mismatch n={n}: {corr}'

    _ = sf.run(build_ghz(n)[0], device='cpu', method='mps', shots=1000, bond_dim=64)
    qc_wu = build_ghz(n)[1]; qc_wu.measure_all()
    _ = qk_mps.run(qc_wu, shots=1000).result()

    def run_sf_m(n=n):
        return sf.run(build_ghz(n)[0], device='cpu', method='mps', shots=10000, bond_dim=64)
    sf_med, sf_times, _, _ = benchmark(run_sf_m)

    def run_qk_m(n=n):
        _, qc = build_ghz(n)
        qc.measure_all()
        return qk_mps.run(qc, shots=10000).result()
    qk_med, qk_times, _, _ = benchmark(run_qk_m)

    mps_results[n] = {'sf_ms': sf_med, 'qk_ms': qk_med, 'sf_all': sf_times, 'qk_all': qk_times, 'correctness': corr}
    print(f'n={n:3d} | SF {sf_med:8.1f}ms | Qiskit {qk_med:8.1f}ms | {fmt(sf_med, qk_med)} | {corr}')

## 4. Shot-based sampling vs Qiskit Aer

Hardware-efficient ansatz (depth 3), statevector method, n = 10–22 x
shots = 1k / 10k / 100k.
**Correctness gate:** statevector fidelity at every size (checked in section 1).

**Caveat:** the speedup is regime-dependent — expect Aer to win at low
shot counts for large n. Read the whole grid, not the best cell.

In [ ]:
shot_qubit_counts = [10, 14, 18, 20, 22]
shot_counts_list = [1000, 10000, 100000]
shot_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print('SHOT-BASED SAMPLING (HE depth=3, Rust-native simulate_and_sample)')
print('=' * 75)
print('Fidelity verified in section 1 (same circuits).\n')

for n_shots in shot_counts_list:
    print(f'--- shots={n_shots} ---')
    for n in shot_qubit_counts:
        _ = sf.run(build_hardware_efficient(n)[0], device='cpu', shots=min(n_shots, 1000), return_statevector=False)
        qc_wu = build_hardware_efficient(n)[1]; qc_wu.measure_all()
        _ = qk_sim.run(qc_wu, shots=min(n_shots, 1000)).result()

        def run_sf_sh(n=n, ns=n_shots):
            return sf.run(build_hardware_efficient(n)[0], device='cpu', shots=ns, return_statevector=False)
        sf_med, sf_times, _, _ = benchmark(run_sf_sh)

        def run_qk_sh(n=n, ns=n_shots):
            _, qc = build_hardware_efficient(n)
            qc.measure_all()
            return qk_sim.run(qc, shots=ns).result()
        qk_med, qk_times, _, _ = benchmark(run_qk_sh)

        shot_results[(n, n_shots)] = {'sf_ms': sf_med, 'qk_ms': qk_med, 'sf_all': sf_times, 'qk_all': qk_times}
        print(f'  n={n:2d} | SF {sf_med:8.1f}ms | Qiskit {qk_med:8.1f}ms | {fmt(sf_med, qk_med)}')

## 5. Depth scaling vs Qiskit Aer

n=16 hardware-efficient ansatz, depths 1–50.

In [ ]:
depth_n = 16
depths = [1, 3, 5, 10, 20, 50]
depth_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print(f'DEPTH SCALING (n={depth_n}, Hardware-Efficient)')
print('=' * 75)

for d in depths:
    sc_c, qc_c = build_hardware_efficient(depth_n, depth=d)
    sf_sv = np.array(sf.run(sc_c, device='cpu', shots=0).statevector)
    qc_c.save_statevector()
    qk_sv = np.array(qk_sim.run(qc_c).result().get_statevector(), dtype=np.complex128)
    fid = fidelity(sf_sv, qk_sv)
    assert fid > 0.9999, f'Depth fidelity fail d={d}: {fid}'

    _ = sf.run(build_hardware_efficient(depth_n, depth=d)[0], device='cpu', shots=0)
    qc_wu = build_hardware_efficient(depth_n, depth=d)[1]; qc_wu.save_statevector()
    _ = qk_sim.run(qc_wu).result()

    def run_sf_d(d=d):
        return sf.run(build_hardware_efficient(depth_n, depth=d)[0], device='cpu', shots=0)
    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf_d)

    def run_qk_d(d=d):
        _, qc = build_hardware_efficient(depth_n, depth=d)
        qc.save_statevector()
        return qk_sim.run(qc).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk_d)

    ng = build_hardware_efficient(depth_n, depth=d)[0].gate_count
    depth_results[d] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'n_gates': ng, 'fidelity': fid,
    }
    print(f'depth={d:2d} ({ng:4d} gates) | SF {sf_med:8.1f}ms | '
          f'Qiskit {qk_med:8.1f}ms | fid={fid:.6f} | {fmt(sf_med, qk_med)}')

## 6. Adjoint gradients — correctness first

n=4, 8 params, TFIM observable. SF adjoint vs PennyLane Lightning adjoint
vs central finite differences. All three must agree before any timing
below means anything.

In [ ]:
from superfermion.qml.gradient.adjoint import adjoint_grad_vector

n_qubits = 4
n_params = 8

H = build_tfim_hamiltonian(n_qubits)
param_names = [f'p{i}' for i in range(n_params)]
sf_params = [sf.param(name) for name in param_names]
c = sf.Circuit(n_qubits)
for i in range(n_qubits):
    c.ry(sf_params[2*i], i)
    c.rz(sf_params[2*i+1], i)
for i in range(n_qubits - 1):
    c.cx(i, i+1)

np.random.seed(42)
vals = np.random.uniform(-np.pi, np.pi, n_params)

sf_grad = adjoint_grad_vector(c, H, param_names, vals)

dev = qml.device('lightning.qubit', wires=n_qubits)
H_pl = qml.Hamiltonian(
    [t.coeffs for t in H.terms],
    [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
     if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
     for t in H.terms]
)

@qml.qnode(dev, diff_method='adjoint')
def pl_circuit(params):
    for i in range(n_qubits):
        qml.RY(params[2*i], wires=i)
        qml.RZ(params[2*i+1], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])
    return qml.expval(H_pl)

from pennylane import numpy as pnp
pl_params = pnp.array(vals, requires_grad=True)
pl_grad = qml.grad(pl_circuit)(pl_params)

eps = 1e-5
fd_grad = np.zeros(n_params)
for i in range(n_params):
    v_plus = vals.copy(); v_plus[i] += eps
    v_minus = vals.copy(); v_minus[i] -= eps
    sv_plus = sf.run(c.bind(dict(zip(param_names, v_plus))), device='cpu', shots=0).statevector
    sv_minus = sf.run(c.bind(dict(zip(param_names, v_minus))), device='cpu', shots=0).statevector
    e_plus = float(np.real(H._fast_expval(sv_plus)))
    e_minus = float(np.real(H._fast_expval(sv_minus)))
    fd_grad[i] = (e_plus - e_minus) / (2 * eps)

print(f'{"param":>6s} {"SF adj":>10s} {"PL adj":>10s} {"FD":>10s}')
print('-' * 42)
for i in range(n_params):
    print(f'{param_names[i]:>6s} {sf_grad[i]:+10.6f} {pl_grad[i]:+10.6f} {fd_grad[i]:+10.6f}')

sf_pl_diff = np.max(np.abs(sf_grad - pl_grad))
sf_fd_diff = np.max(np.abs(sf_grad - fd_grad))
print(f'\nMax |SF - PL|: {sf_pl_diff:.2e}')
print(f'Max |SF - FD|: {sf_fd_diff:.2e}')
assert sf_pl_diff < 1e-6 and sf_fd_diff < 1e-4, 'Gradient agreement failed'
print('GRADIENTS AGREE. Timing below is meaningful.')

## 7. Adjoint vs PennyLane Lightning — scaling with qubit count

HE ansatz depth 1, TFIM observable, n = 4–18. SF `adjoint_grad_vector()`
(one Python->Rust FFI call) vs PL `qml.grad` on `lightning.qubit`
(adjoint diff method).

**Caveat:** at small n the comparison is dominated by dispatch overhead
(one FFI call vs a full autograd transform). At n >= 18 PennyLane is
expected to win — read the trend, not the biggest number.

In [ ]:
qubit_sizes = [4, 6, 8, 10, 12, 14, 16, 18]
grad_results = []

print('='*80)
print('ADJOINT GRADIENT BENCHMARK (HE ansatz, depth=1, TFIM observable)')
print('='*80)

def pl_hamiltonian(H_n):
    return qml.Hamiltonian(
        [t.coeffs for t in H_n.terms],
        [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
         if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
         for t in H_n.terms]
    )

print(f'Warming up at n={qubit_sizes[-1]}...')
n_warm = qubit_sizes[-1]
npar_warm = 2 * n_warm
H_warm = build_tfim_hamiltonian(n_warm)
pn_warm = [f'p{i}' for i in range(npar_warm)]
sp_warm = [sf.param(nm) for nm in pn_warm]
c_warm = sf.Circuit(n_warm)
for i in range(n_warm):
    c_warm.ry(sp_warm[2*i], i)
    c_warm.rz(sp_warm[2*i+1], i)
for i in range(n_warm - 1):
    c_warm.cx(i, i+1)
v_warm = np.random.uniform(-np.pi, np.pi, npar_warm)
adjoint_grad_vector(c_warm, H_warm, pn_warm, v_warm)

dev_warm = qml.device('lightning.qubit', wires=n_warm)
H_pl_warm = pl_hamiltonian(H_warm)

@qml.qnode(dev_warm, diff_method='adjoint')
def pl_warm_fn(params):
    for i in range(n_warm):
        qml.RY(params[2*i], wires=i)
        qml.RZ(params[2*i+1], wires=i)
    for i in range(n_warm - 1):
        qml.CNOT(wires=[i, i+1])
    return qml.expval(H_pl_warm)

qml.grad(pl_warm_fn)(pnp.array(v_warm, requires_grad=True))
print('Done.\n')

for n in qubit_sizes:
    n_par = 2 * n
    H_n = build_tfim_hamiltonian(n)
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n)
    for i in range(n):
        circ.ry(sparams[2*i], i)
        circ.rz(sparams[2*i+1], i)
    for i in range(n - 1):
        circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    dev_n = qml.device('lightning.qubit', wires=n)
    H_pl_n = pl_hamiltonian(H_n)

    @qml.qnode(dev_n, diff_method='adjoint')
    def pl_fn(params):
        for i in range(n):
            qml.RY(params[2*i], wires=i)
            qml.RZ(params[2*i+1], wires=i)
        for i in range(n - 1):
            qml.CNOT(wires=[i, i+1])
        return qml.expval(H_pl_n)

    pl_p = pnp.array(vv, requires_grad=True)

    adjoint_grad_vector(circ, H_n, pnames, vv)
    qml.grad(pl_fn)(pl_p)

    def run_sf():
        return adjoint_grad_vector(circ, H_n, pnames, vv)
    sf_med, _, sf_q1, sf_q3 = benchmark(run_sf)

    def run_pl():
        return qml.grad(pl_fn)(pl_p)
    pl_med, _, pl_q1, pl_q3 = benchmark(run_pl)

    grad_results.append({
        'n': n, 'n_params': n_par,
        'sf_med': sf_med, 'sf_q1': sf_q1, 'sf_q3': sf_q3,
        'pl_med': pl_med, 'pl_q1': pl_q1, 'pl_q3': pl_q3,
    })
    print(f'n={n:2d} ({n_par:2d} params) | '
          f'SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'PL {pl_med:8.1f}ms [{pl_q1:.1f}-{pl_q3:.1f}] | '
          f'{fmt(sf_med, pl_med)}')

## 8. Adjoint — scaling with depth (fixed n=10)

Depth 1–12. The adjoint property being demonstrated: gradient cost is
independent of parameter count.

In [ ]:
n_fixed = 10
depths_adj = [1, 2, 3, 5, 8, 12]
adj_depth_results = []

print('='*80)
print(f'ADJOINT DEPTH SCALING (n={n_fixed}, TFIM)')
print('='*80)

H_fixed = build_tfim_hamiltonian(n_fixed)
H_pl_fixed = pl_hamiltonian(H_fixed)

for depth in depths_adj:
    n_par = 2 * n_fixed * depth
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n_fixed)
    pidx = 0
    for _ in range(depth):
        for i in range(n_fixed):
            circ.ry(sparams[pidx], i); pidx += 1
            circ.rz(sparams[pidx], i); pidx += 1
        for i in range(n_fixed - 1):
            circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    dev_d = qml.device('lightning.qubit', wires=n_fixed)

    @qml.qnode(dev_d, diff_method='adjoint')
    def pl_depth_fn(params):
        pidx = 0
        for _ in range(depth):
            for i in range(n_fixed):
                qml.RY(params[pidx], wires=i); pidx += 1
                qml.RZ(params[pidx], wires=i); pidx += 1
            for i in range(n_fixed - 1):
                qml.CNOT(wires=[i, i+1])
        return qml.expval(H_pl_fixed)

    pl_p = pnp.array(vv, requires_grad=True)

    adjoint_grad_vector(circ, H_fixed, pnames, vv)
    qml.grad(pl_depth_fn)(pl_p)

    def run_sf():
        return adjoint_grad_vector(circ, H_fixed, pnames, vv)
    sf_med, _, sf_q1, sf_q3 = benchmark(run_sf)

    def run_pl():
        return qml.grad(pl_depth_fn)(pl_p)
    pl_med, _, pl_q1, pl_q3 = benchmark(run_pl)

    adj_depth_results.append({
        'd': depth, 'n_params': n_par,
        'sf_med': sf_med, 'pl_med': pl_med,
    })
    print(f'd={depth:2d} ({n_par:3d} params) | '
          f'SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'PL {pl_med:8.1f}ms [{pl_q1:.1f}-{pl_q3:.1f}] | '
          f'{fmt(sf_med, pl_med)}')

## 9. Adjoint vs parameter-shift (SF internal)

The expected algorithmic property of adjoint differentiation
(Jones & Gacon 2020): one backward sweep regardless of parameter count,
vs parameter-shift's 2N forward passes. n=10, depths 1–8.

In [ ]:
from superfermion.qml.gradient.parameter_shift import parameter_shift_grad_vector

print('='*80)
print('ADJOINT vs PARAMETER-SHIFT (SF, n=10, TFIM)')
print('='*80)

n_test = 10
H_test = build_tfim_hamiltonian(n_test)
adj_vs_ps_results = []

for depth in [1, 2, 4, 8]:
    n_par = 2 * n_test * depth
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n_test)
    pidx = 0
    for _ in range(depth):
        for i in range(n_test):
            circ.ry(sparams[pidx], i); pidx += 1
            circ.rz(sparams[pidx], i); pidx += 1
        for i in range(n_test - 1):
            circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    adjoint_grad_vector(circ, H_test, pnames, vv)
    parameter_shift_grad_vector(circ, H_test, pnames, vv)

    def run_adj():
        return adjoint_grad_vector(circ, H_test, pnames, vv)
    def run_ps():
        return parameter_shift_grad_vector(circ, H_test, pnames, vv)

    adj_med, _, _, _ = benchmark(run_adj)
    ps_med, _, _, _ = benchmark(run_ps)

    adj_vs_ps_results.append({'depth': depth, 'n_params': n_par, 'adj': adj_med, 'ps': ps_med})
    print(f'd={depth:2d} ({n_par:3d} params) | '
          f'Adjoint {adj_med:8.2f}ms | '
          f'Param-shift {ps_med:8.1f}ms | '
          f'Adjoint {ps_med/adj_med:.1f}x faster')

## 10. Transpilation vs Qiskit — `run_benchpress.py`

100-qubit circuits (QFT, QV, SU2, BV, Heisenberg, QAOA,
simplification, Clifford) on heavy-hex-127 coupling, native basis
{rz, sx, x, ecr}. Wall-clock AND 2Q gate counts both recorded.

**Caveat:** SF transpiles faster in wall-clock but emits more 2Q gates —
speed at the cost of output quality. The full results land in
`benchpress_post_gaps.json`, which is the credibility file: it records
where SF loses.

In [ ]:
print('Running run_benchpress.py (100Q transpilation suite)...')
print('This section uses 180s-per-side subprocess timeouts; it can take a while.')
print('='*80)

proc = subprocess.run(
    ['python', 'run_benchpress.py'],
    capture_output=True, text=True, timeout=3600,
)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:', proc.stderr[-2000:])

benchpress_results = json.loads(Path('benchpress_post_gaps.json').read_text())
print(f'\nLoaded {len(benchpress_results)} benchpress results.')

## 11. Scoreboard — every claim, this machine, this run

The table regenerates from the live results above. If a number in the
README disagrees with this table, this table wins.

In [ ]:
def rng(xs): return (min(xs), max(xs))

sv_ratios = [v['qk_ms']/v['sf_ms'] for v in sv_results.values()]
stab_ratios = [v['qk_ms']/v['sf_ms'] for v in stab_results.values()]
mps_ratios = [v['qk_ms']/v['sf_ms'] for v in mps_results.values()]
shot_ratios = [v['qk_ms']/v['sf_ms'] for v in shot_results.values()]
adj_ps_ratios = [r['ps']/r['adj'] for r in adj_vs_ps_results]
adj_pl_ratios = [r['pl_med']/r['sf_med'] for r in grad_results]

bp_sf_wins = sum(1 for r in benchpress_results if r.get('winner') == 'SF')
bp_total = len(benchpress_results)

rows = [
    ('Statevector vs Aer (n=10-22)',
     f'{rng(sv_ratios)[0]:.2f}-{rng(sv_ratios)[1]:.2f}x',
     'regime-dependent; see section 1'),
    ('Stabilizer vs Aer (n=10-500)',
     f'{rng(stab_ratios)[0]:.1f}-{rng(stab_ratios)[1]:.1f}x', ''),
    ('MPS GHZ vs Aer (n=10-100)',
     f'{rng(mps_ratios)[0]:.0f}-{rng(mps_ratios)[1]:.0f}x', 'best case for MPS'),
    ('Shot sampling vs Aer (100k shots)',
     f'{rng(shot_ratios)[0]:.1f}-{rng(shot_ratios)[1]:.1f}x', 'regime-dependent'),
    ('Adjoint vs param-shift (n=10)',
     f'{rng(adj_ps_ratios)[0]:.0f}-{rng(adj_ps_ratios)[1]:.0f}x', 'expected algorithmic property'),
    ('Adjoint vs PennyLane (n=4-16)',
     f'{rng(adj_pl_ratios)[0]:.1f}-{rng(adj_pl_ratios)[1]:.0f}x',
     f'PL faster at n={grad_results[-1]["n"]}' if adj_pl_ratios[-1] < 1 else ''),
    ('Transpilation wall-clock (100Q)',
     f'SF wins {bp_sf_wins}/{bp_total} tests', 'more 2Q gates emitted'),
]

print(f'Machine: {PLATFORM["cpu_model"]} ({PLATFORM["logical_cores"]} threads)')
print(f'Run: {PLATFORM["run_date"]}  |  SF {PLATFORM["superfermion"]} vs Qiskit {PLATFORM["qiskit"]} / PL {PLATFORM["pennylane"]}')
print()
print(f'| Benchmark | Result | Caveat |')
print(f'|---|---|---|')
for name, res, cav in rows:
    print(f'| {name} | {res} | {cav} |')

# ── persist everything as the canonical data file ──
all_data = {
    'platform': PLATFORM,
    'statevector': {str(k): v for k, v in sv_results.items()},
    'stabilizer': {str(k): v for k, v in stab_results.items()},
    'mps': {str(k): v for k, v in mps_results.items()},
    'shots': {str(k): v for k, v in shot_results.items()},
    'depth': {str(k): v for k, v in depth_results.items()},
    'adjoint_vs_pennylane': grad_results,
    'adjoint_depth_scaling': adj_depth_results,
    'adjoint_vs_paramshift': adj_vs_ps_results,
    'transpilation': benchpress_results,
}
with open('benchmark_data.json', 'w') as f:
    json.dump(all_data, f, indent=2, default=str)
print('\nSaved benchmark_data.json — this is the canonical benchmark data file.')